# SPARK Tutorial 2026
## Brain Tumor Segmentation with U-Net (TensorFlow / Keras)

<center><img src="https://www.mayoclinic.org/-/media/kcms/gbs/patient-consumer/images/2014/10/30/15/17/mcdc7_brain_cancer-8col.jpg" alt="Brain MRI" style='width:350px;'></center>

**Image source**: Mayo Clinic

## About This Tutorial

This hands-on tutorial walks you through **brain tumor segmentation** using the classic **U-Net** architecture implemented in TensorFlow/Keras. We will:

1. **Understand** the clinical motivation for automated brain tumor segmentation
2. **Learn** key image segmentation concepts from the ground up
3. **Explore** evaluation metrics (Dice, IoU, Accuracy)
4. **Deep-dive** into the U-Net architecture layer by layer
5. **Survey** state-of-the-art models: nnU-Net, Swin UNETR, MedSAM, TransBTS, and Attention U-Net
6. **Implement** a complete segmentation pipeline on the LGG MRI dataset (Kaggle)
7. **Evaluate** and **visualize** model predictions

> **Environment**: This notebook is designed to run on **Kaggle** with GPU acceleration.
> **Dataset**: [LGG MRI Segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation) (Brain MRI images with manual FLAIR abnormality segmentation masks).

**References**:
- Lee et al., *Data in Brief*, 2024. [DOI:10.1016/j.dib.2024.111159](https://doi.org/10.1016/j.dib.2024.111159)
- Ronneberger et al., *U-Net: Convolutional Networks for Biomedical Image Segmentation*, MICCAI 2015. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)

# <p style="background-color:#2c3e50;color:white;font-size:22px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Table of Contents</p>

1. [Part 1 — Background & Motivation](#part1)
    - [1.1 What is a Brain Tumor?](#sec1_1)
    - [1.2 Medical Imaging & MRI](#sec1_2)
    - [1.3 What is Image Segmentation?](#sec1_3)
    - [1.4 Why Segmentation for Brain Tumors?](#sec1_4)
2. [Part 2 — Segmentation Metrics Deep Dive](#part2)
    - [2.1 Dice Coefficient (F1 Score)](#sec2_1)
    - [2.2 Intersection over Union (IoU / Jaccard)](#sec2_2)
    - [2.3 Pixel Accuracy](#sec2_3)
3. [Part 3 — U-Net Architecture Deep Dive](#part3)
    - [3.1 History & Motivation](#sec3_1)
    - [3.2 Encoder (Contracting Path)](#sec3_2)
    - [3.3 Bottleneck](#sec3_3)
    - [3.4 Decoder (Expanding Path)](#sec3_4)
    - [3.5 Skip Connections](#sec3_5)
4. [Part 4 — State-of-the-Art Models for Brain Tumor Segmentation](#part4)
    - [4.1 nnU-Net](#sec4_1)
    - [4.2 Swin UNETR](#sec4_2)
    - [4.3 MedSAM](#sec4_3)
    - [4.4 TransBTS](#sec4_4)
    - [4.5 Attention U-Net](#sec4_5)
    - [4.6 Comparison Table](#sec4_6)
5. [Part 5 — Hands-On Implementation](#part5)
    - [5.1 Setup & Imports](#sec5_1)
    - [5.2 Data Loading & Exploration](#sec5_2)
    - [5.3 Data Preprocessing & Augmentation](#sec5_3)
    - [5.4 Build U-Net Model](#sec5_4)
    - [5.5 Loss Functions & Metrics](#sec5_5)
    - [5.6 Compile & Train](#sec5_6)
    - [5.7 Training History Visualization](#sec5_7)
    - [5.8 Model Evaluation](#sec5_8)
    - [5.9 Prediction Visualization](#sec5_9)
6. [Part 6 — Summary & Next Steps](#part6)
7. [References](#refs)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 1 — Background & Motivation</p> <a id="part1"></a>

## <span style="color:#1a5276;font-size:22px">1.1 What is a Brain Tumor?</span> <a id="sec1_1"></a>

A **brain tumor** is an abnormal growth of cells within the brain or the central spinal canal. Brain tumors are broadly classified as:

| Category | Description | Examples |
|----------|-------------|----------|
| **Benign** | Non-cancerous, slow-growing, often have clear boundaries | Meningiomas, Schwannomas |
| **Malignant** | Cancerous, fast-growing, invade surrounding tissue | Glioblastoma (GBM), Anaplastic Astrocytoma |
| **Primary** | Originate in the brain | Gliomas, Medulloblastomas |
| **Secondary (Metastatic)** | Spread from cancer elsewhere in the body | Lung/breast cancer metastases |

**Gliomas** are the most common primary brain tumors (about 30% of all brain tumors, 80% of malignant ones). They arise from **glial cells** and are graded by the WHO from Grade I (least aggressive) to Grade IV (most aggressive — glioblastoma).

### Why Automated Segmentation Matters
- **Manual segmentation** by radiologists is time-consuming (30-60 minutes per scan) and subject to inter-observer variability.
- **Automated segmentation** enables faster diagnosis, treatment planning, and longitudinal monitoring.
- It is essential for **surgical planning**, **radiotherapy targeting**, and **treatment response assessment**.

## <span style="color:#1a5276;font-size:22px">1.2 Medical Imaging & MRI</span> <a id="sec1_2"></a>

**Magnetic Resonance Imaging (MRI)** is the gold standard for brain tumor imaging because it provides excellent soft-tissue contrast without ionizing radiation.

### Common MRI Sequences for Brain Tumors

| Sequence | Full Name | What It Shows |
|----------|-----------|---------------|
| **T1** | T1-weighted | Anatomy; tumors appear dark |
| **T1ce** | T1 with Contrast Enhancement (Gadolinium) | Enhancing tumor regions (active tumor) appear bright |
| **T2** | T2-weighted | Edema and tumors appear bright |
| **FLAIR** | Fluid-Attenuated Inversion Recovery | Like T2 but suppresses CSF signal; highlights perilesional edema |

In this tutorial, we work with **FLAIR abnormality** segmentation masks from the LGG (Lower-Grade Glioma) MRI dataset, which contains 2D brain MRI slices paired with manual segmentation masks.

## <span style="color:#1a5276;font-size:22px">1.3 What is Image Segmentation?</span> <a id="sec1_3"></a>

**Image segmentation** is the task of assigning a class label to every pixel in an image, effectively partitioning the image into meaningful regions.

### Types of Image Segmentation

| Type | Description | Example |
|------|-------------|---------|
| **Semantic Segmentation** | Classify every pixel into a class (no instance distinction) | All tumor pixels → "tumor" class |
| **Instance Segmentation** | Distinguish individual objects of the same class | Separate tumor 1 from tumor 2 |
| **Panoptic Segmentation** | Combines semantic + instance segmentation | Full scene understanding |

For brain tumor segmentation, we typically use **semantic segmentation** — each pixel is classified as either **tumor** or **background** (binary) or into sub-regions like **enhancing tumor**, **tumor core**, and **whole tumor** (multi-class).

### How It Differs from Classification & Detection

```
Classification:  "This image contains a tumor"          → Single label
Detection:       "There is a tumor at (x, y, w, h)"     → Bounding box
Segmentation:    "These exact pixels are tumor"          → Pixel-level mask
```

Segmentation provides the most precise spatial information, which is critical for surgical planning and dosimetry in radiation therapy.

## <span style="color:#1a5276;font-size:22px">1.4 Why Segmentation for Brain Tumors?</span> <a id="sec1_4"></a>

| Application | How Segmentation Helps |
|-------------|----------------------|
| **Surgical Planning** | Precise tumor boundaries guide neurosurgeons to maximize resection while preserving eloquent cortex |
| **Radiation Therapy** | Accurate contours ensure radiation targets the tumor and spares healthy tissue |
| **Treatment Monitoring** | Volumetric measurements from serial scans track tumor growth or shrinkage |
| **Prognosis** | Tumor volume, shape, and heterogeneity correlate with patient outcomes |
| **Clinical Trials** | Standardized segmentation enables objective, reproducible endpoints |

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 2 — Segmentation Metrics Deep Dive</p> <a id="part2"></a>

Understanding evaluation metrics is essential before building any segmentation model. In medical imaging, choosing the right metric is especially important because class imbalance is severe (tumor pixels are typically a small fraction of the image).

## <span style="color:#1a5276;font-size:22px">2.1 Dice Coefficient (F1 Score)</span> <a id="sec2_1"></a>

The **Dice Similarity Coefficient (DSC)** measures the overlap between the predicted segmentation $P$ and the ground truth $G$:

$$\text{Dice}(P, G) = \frac{2 \times |P \cap G|}{|P| + |G|}$$

- **Range**: 0 (no overlap) to 1 (perfect overlap)
- **Interpretation**: Harmonic mean of precision and recall at the pixel level
- **Why it's popular in medical imaging**: It is robust to class imbalance because it focuses on the positive (tumor) class
- **Smooth variant** (used in practice to avoid division by zero):

$$\text{Dice}_{\text{smooth}} = \frac{2 \times \sum(P \cdot G) + \epsilon}{\sum P + \sum G + \epsilon}$$

where $\epsilon$ is a small smoothing constant (e.g., 1 or 100).

### Dice Loss

Since we want to **maximize** Dice but neural networks **minimize** loss, we define:

$$\mathcal{L}_{\text{Dice}} = 1 - \text{Dice}(P, G)$$

or equivalently, the negative Dice coefficient (as used in our implementation).

## <span style="color:#1a5276;font-size:22px">2.2 Intersection over Union (IoU / Jaccard Index)</span> <a id="sec2_2"></a>

The **IoU** (also called the Jaccard Index) measures the ratio of the intersection to the union of the predicted and ground truth regions:

$$\text{IoU}(P, G) = \frac{|P \cap G|}{|P \cup G|} = \frac{|P \cap G|}{|P| + |G| - |P \cap G|}$$

- **Range**: 0 to 1
- **Relationship to Dice**: $\text{Dice} = \frac{2 \times \text{IoU}}{1 + \text{IoU}}$
- IoU penalizes false positives and false negatives more harshly than Dice
- **BraTS Challenge** uses both Dice and Hausdorff Distance as primary metrics

## <span style="color:#1a5276;font-size:22px">2.3 Pixel Accuracy</span> <a id="sec2_3"></a>

$$\text{Pixel Accuracy} = \frac{\text{Correctly classified pixels}}{\text{Total pixels}}$$

**Caution**: Pixel accuracy can be misleading with imbalanced classes. If only 2% of pixels are tumor, a model that predicts "no tumor" everywhere achieves 98% accuracy but is clinically useless. This is why **Dice and IoU** are preferred for segmentation evaluation.

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 3 — U-Net Architecture Deep Dive</p> <a id="part3"></a>

<center><img src="https://miro.medium.com/max/1200/1*f7YOaE4TWubwaFF7Z1fzNw.png" alt="U-Net Architecture" style='width:800px;'></center>

*The U-Net architecture. Left: contracting (encoder) path. Right: expanding (decoder) path. Gray arrows: skip connections. (Ronneberger et al., 2015)*

## <span style="color:#1a5276;font-size:22px">3.1 History & Motivation</span> <a id="sec3_1"></a>

**U-Net** was introduced by Olaf Ronneberger, Philipp Fischer, and Thomas Brox in 2015 at MICCAI. It was designed specifically for **biomedical image segmentation** where:

- **Training data is scarce** — medical images are expensive to annotate
- **Precise localization is required** — every pixel matters for clinical decisions
- **Context and detail must coexist** — the model needs both global understanding and fine-grained boundaries

### Key Innovations
1. **Symmetric encoder-decoder** with a U-shaped structure
2. **Skip connections** that concatenate encoder features with decoder features, preserving spatial detail
3. **Data augmentation** strategies (elastic deformations) for medical images
4. **Few training images** — originally demonstrated with only 30 annotated images

### Architecture Overview

```
Input Image (256×256×3)
    │
    ▼
┌─────────────────────────────┐
│     ENCODER (Contracting)    │   Captures WHAT is in the image
│  Conv → Conv → Pool  (×4)   │   Spatial resolution ↓, Channels ↑
└─────────┬───────────────────┘
          │
          ▼
┌─────────────────────────────┐
│        BOTTLENECK            │   Deepest representation
│     Conv → Conv (1024 ch)    │   (16×16 for 256×256 input)
└─────────┬───────────────────┘
          │
          ▼
┌─────────────────────────────┐
│     DECODER (Expanding)      │   Recovers WHERE things are
│  UpConv → Concat → Conv (×4)│   Spatial resolution ↑, Channels ↓
└─────────┬───────────────────┘
          │
          ▼
    Output Mask (256×256×1)
```

## <span style="color:#1a5276;font-size:22px">3.2 Encoder (Contracting Path)</span> <a id="sec3_2"></a>

The encoder follows a typical CNN pattern. At each level:

1. **Two 3×3 convolutions** (each followed by ReLU activation and batch normalization)
2. **2×2 max pooling** with stride 2 for downsampling

```
Level 1:  Input(256×256×3)  → Conv(64) → Conv(64)  → Pool → (128×128×64)
Level 2:  (128×128×64)      → Conv(128)→ Conv(128) → Pool → (64×64×128)
Level 3:  (64×64×128)       → Conv(256)→ Conv(256) → Pool → (32×32×256)
Level 4:  (32×32×256)       → Conv(512)→ Conv(512) → Pool → (16×16×512)
```

**What the encoder learns**: At each deeper level, the network captures increasingly abstract features:
- **Level 1**: Edges, textures
- **Level 2**: Simple shapes, patterns
- **Level 3**: Object parts
- **Level 4**: High-level semantic concepts (e.g., "tumor-like region")

The **trade-off**: As spatial resolution decreases, the model gains a larger **receptive field** (sees more context) but loses fine spatial details. This is where the decoder and skip connections come in.

## <span style="color:#1a5276;font-size:22px">3.3 Bottleneck</span> <a id="sec3_3"></a>

The bottleneck is the deepest part of the network:

```
Bottleneck: (16×16×512) → Conv(1024) → Conv(1024) → (16×16×1024)
```

- Acts as the **bridge** between encoder and decoder
- Contains the most compressed, highest-level representation of the input
- Has the **largest receptive field** — each "pixel" in the bottleneck "sees" the entire input image
- No pooling is applied here — the spatial dimensions are preserved

## <span style="color:#1a5276;font-size:22px">3.4 Decoder (Expanding Path)</span> <a id="sec3_4"></a>

The decoder reconstructs the spatial resolution. At each level:

1. **2×2 transposed convolution** (up-convolution) to upsample the feature map
2. **Concatenation** with the corresponding encoder feature map (skip connection)
3. **Two 3×3 convolutions** (each followed by ReLU and batch normalization)

```
Level 4:  UpConv(16→32) + Skip(32×32×256) → Conv(512) → Conv(512)  → (32×32×512)
Level 3:  UpConv(32→64) + Skip(64×64×128) → Conv(256) → Conv(256)  → (64×64×256)
Level 2:  UpConv(64→128)+ Skip(128×128×64)→ Conv(128) → Conv(128)  → (128×128×128)
Level 1:  UpConv(128→256)+ Skip(256×256×64)→ Conv(64) → Conv(64)   → (256×256×64)
```

**Final layer**: A 1×1 convolution maps the 64-channel feature map to the output segmentation mask with sigmoid activation:
```
(256×256×64) → Conv1×1(1) → Sigmoid → (256×256×1)
```

## <span style="color:#1a5276;font-size:22px">3.5 Skip Connections — The Secret Sauce</span> <a id="sec3_5"></a>

Skip connections are what make U-Net special. They **concatenate** feature maps from the encoder directly to the decoder at the same spatial level.

### Why Are They Important?

| Without Skip Connections | With Skip Connections |
|--------------------------|----------------------|
| Decoder must reconstruct spatial details from compressed representation alone | Decoder receives both high-level semantics AND low-level spatial details |
| Blurry, imprecise boundaries | Sharp, precise boundaries |
| Gradient flow only through bottleneck | Additional gradient pathways improve training |

### How They Work

```
Encoder Level 2 output: (128×128×128) ─────────────┐
        │                                            │
        ▼ (MaxPool)                                  │ (Skip Connection)
  [deeper levels...]                                 │
        │                                            │
        ▼ (UpConv to 128×128×512)                    │
        └──────── CONCATENATE ───────────────────────┘
                        │
                        ▼
              (128×128×640)  ← combined channels
                        │
                   Conv → Conv
                        │
                        ▼
              (128×128×128)
```

This allows the decoder to have access to both:
- **"What"**: High-level features from the bottleneck (semantic understanding)
- **"Where"**: Fine-grained spatial features from the encoder (precise localization)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 4 — State-of-the-Art Models for Brain Tumor Segmentation</p> <a id="part4"></a>

While U-Net remains the foundation for medical image segmentation, several advanced architectures have achieved state-of-the-art results on brain tumor segmentation benchmarks such as the **BraTS Challenge** (Brain Tumor Segmentation Challenge). Below we survey five influential models.

## <span style="color:#1a5276;font-size:22px">4.1 nnU-Net (No-New-UNet)</span> <a id="sec4_1"></a>

### Overview
**nnU-Net** is not a new architecture but a **self-configuring framework** that automatically adapts U-Net-based architectures to any given medical segmentation dataset. It was introduced by Fabian Isensee et al. from the German Cancer Research Center (DKFZ).

### Architecture Flow

```
   Input Dataset
        │
        ▼
┌──────────────────────────────────┐
│   AUTOMATED CONFIGURATION        │
│                                  │
│  1. Dataset Fingerprinting       │  ← Analyzes image sizes, spacing,
│     (image stats, class ratios)  │     intensity distributions
│                                  │
│  2. Pipeline Selection           │  ← 2D U-Net, 3D full-res U-Net,
│     (architecture + config)      │     or 3D cascade U-Net
│                                  │
│  3. Preprocessing Rules          │  ← Resampling, normalization,
│     (auto-determined)            │     cropping strategies
│                                  │
│  4. Training Scheme              │  ← Loss (Dice + CE), optimizer,
│     (auto-determined)            │     augmentation, scheduling
│                                  │
│  5. Post-processing              │  ← Connected component analysis,
│     (auto-determined)            │     ensemble strategies
└──────────────────────────────────┘
        │
        ▼
   Optimized Segmentation
```

### Key Innovations
- **No manual tuning**: The framework fingerprints the dataset and automatically selects preprocessing, architecture, training scheme, and post-processing
- **Three U-Net configurations**: 2D, 3D full-resolution, and 3D cascade (coarse-to-fine)
- **Robust augmentation**: Rotation, scaling, mirroring, elastic deformation, gamma correction
- **Ensemble of models**: Combines predictions from multiple configurations for best results

### Performance
- **BraTS 2020**: Achieved top results without any task-specific modifications
- **Winner/top performer** in 33+ medical segmentation challenges
- Consistently outperforms hand-tuned architectures across diverse tasks

### Reference
- Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. "nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation." *Nature Methods*, 18, 203–211 (2021). [DOI:10.1038/s41592-020-01008-z](https://doi.org/10.1038/s41592-020-01008-z)
- GitHub: [https://github.com/MIC-DKFZ/nnUNet](https://github.com/MIC-DKFZ/nnUNet)

## <span style="color:#1a5276;font-size:22px">4.2 Swin UNETR</span> <a id="sec4_2"></a>

### Overview
**Swin UNETR** (Swin UNEt TRansformer) replaces the CNN encoder with a **Swin Transformer** while keeping the U-Net decoder structure. It was developed as part of the MONAI project and won the **BraTS 2021 Challenge**.

### Architecture Flow

```
   Input Volume (4-channel 3D MRI: T1, T1ce, T2, FLAIR)
        │
        ▼
┌────────────────────────────────────┐
│     SWIN TRANSFORMER ENCODER       │
│                                    │
│  Patch Partition (4×4×4 patches)   │
│        │                           │
│        ▼                           │
│  Stage 1: Swin Transformer Block   │──→ Skip to Decoder
│    (Window Multi-Head Self-Attn)   │
│    (Shifted Window Self-Attn)      │
│        │ Patch Merging             │
│        ▼                           │
│  Stage 2: Swin Transformer Block   │──→ Skip to Decoder
│        │ Patch Merging             │
│        ▼                           │
│  Stage 3: Swin Transformer Block   │──→ Skip to Decoder
│        │ Patch Merging             │
│        ▼                           │
│  Stage 4: Swin Transformer Block   │──→ Skip to Decoder
└────────────────┬───────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│         BOTTLENECK                 │
│  (Deepest Swin Transformer stage)  │
└────────────────┬───────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│     CNN DECODER (U-Net style)      │
│                                    │
│  Deconv → Concat(Skip) → Conv(×4) │
│        │                           │
│        ▼                           │
│  1×1×1 Conv → Softmax             │
└────────────────────────────────────┘
        │
        ▼
   Multi-class Segmentation
   (Enhancing Tumor, Tumor Core, Whole Tumor)
```

### Key Innovations
- **Shifted Window Self-Attention**: Efficiently computes attention within local windows, then shifts windows to enable cross-window connections — achieving global context with linear complexity
- **Hierarchical feature maps**: Like a CNN, features at multiple resolutions feed into the decoder via skip connections
- **Self-supervised pre-training**: Pre-trained on 5,050 unlabeled CT scans, significantly boosting performance on downstream tasks
- **3D native**: Processes volumetric data directly (not slice-by-slice)

### Performance
- **BraTS 2021 Winner**: Dice scores of 91.6% (Whole Tumor), 86.5% (Tumor Core), 83.1% (Enhancing Tumor)
- Demonstrates that transformers can outperform pure CNN approaches in 3D medical segmentation

### Reference
- Hatamizadeh, A. et al. "Swin UNETR: Swin Transformers for Semantic Segmentation of Brain Tumours in MRI Images." *BrainLes Workshop, MICCAI 2021*. [arXiv:2201.01266](https://arxiv.org/abs/2201.01266)
- Tang, Y. et al. "Self-Supervised Pre-Training of Swin Transformers for 3D Medical Image Analysis." *CVPR 2022*. [arXiv:2111.14791](https://arxiv.org/abs/2111.14791)
- GitHub: [https://github.com/Project-MONAI/tutorials](https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/swin_unetr_brats21_segmentation_3d.ipynb)

## <span style="color:#1a5276;font-size:22px">4.3 MedSAM (Medical Segment Anything Model)</span> <a id="sec4_3"></a>

### Overview
**MedSAM** adapts Meta's **Segment Anything Model (SAM)** to the medical imaging domain. SAM was originally trained on 11 million natural images; MedSAM fine-tunes it on over 1.5 million medical image-mask pairs across 10+ imaging modalities.

### Architecture Flow

```
   Input Medical Image (MRI slice)
        │
        ▼
┌────────────────────────────────────┐
│     IMAGE ENCODER (ViT-B/H)       │
│                                    │
│  Pre-trained Vision Transformer    │
│  (from SAM, fine-tuned on medical) │
│  Patch Embedding → Transformer     │
│  Blocks → Image Embeddings         │
└────────────────┬───────────────────┘
                 │
                 │    ┌──────────────────────────────┐
                 │    │     PROMPT ENCODER            │
                 │    │                              │
                 │    │  Bounding Box / Points /     │
                 │    │  Text → Prompt Embeddings    │
                 │    └──────────────┬───────────────┘
                 │                   │
                 ▼                   ▼
┌────────────────────────────────────┐
│     MASK DECODER                   │
│                                    │
│  Cross-Attention between image     │
│  embeddings and prompt embeddings  │
│        │                           │
│  Lightweight MLP → Upsampling      │
│        │                           │
│  Output: Segmentation Mask(s)      │
└────────────────────────────────────┘
        │
        ▼
   Binary Segmentation Mask
```

### Key Innovations
- **Foundation model paradigm**: One pre-trained model works across multiple organs, modalities, and tasks with minimal prompting
- **Interactive segmentation**: Users provide bounding boxes or point prompts to guide segmentation — especially useful for clinical workflows
- **Massive training data**: Fine-tuned on 1.5M+ medical image-mask pairs spanning CT, MRI, X-ray, ultrasound, endoscopy, dermoscopy, and more
- **Zero-shot generalization**: Can segment unseen organs/pathologies with appropriate prompts

### Limitations
- **Prompt-dependent**: Requires user input (bounding box) — not fully automatic
- **2D only**: Processes slices independently; lacks 3D volumetric reasoning
- **Variable performance**: Excellent for well-defined structures, less reliable for diffuse or ambiguous boundaries

### Reference
- Ma, J. et al. "Segment Anything in Medical Images." *Nature Communications*, 15, 654 (2024). [DOI:10.1038/s41467-024-44824-z](https://doi.org/10.1038/s41467-024-44824-z)
- GitHub (MedSAM): [https://github.com/bowang-lab/MedSAM](https://github.com/bowang-lab/MedSAM)
- GitHub (Brain-specific): [https://github.com/vpulab/med-sam-brain](https://github.com/vpulab/med-sam-brain)

## <span style="color:#1a5276;font-size:22px">4.4 TransBTS (Transformer for Brain Tumor Segmentation)</span> <a id="sec4_4"></a>

### Overview
**TransBTS** was one of the first architectures to introduce **Transformers into brain tumor segmentation**. It uses a 3D CNN encoder to extract local features, then passes them through a Transformer module to capture global context, followed by a CNN decoder.

### Architecture Flow

```
   Input 3D MRI Volume (4 modalities)
        │
        ▼
┌────────────────────────────────────┐
│     3D CNN ENCODER                 │
│                                    │
│  Stage 1: 3D Conv → BN → ReLU     │──→ Skip
│        │ (Downsample)              │
│  Stage 2: 3D Conv → BN → ReLU     │──→ Skip
│        │ (Downsample)              │
│  Stage 3: 3D Conv → BN → ReLU     │──→ Skip
│        │ (Downsample)              │
│  Stage 4: 3D Conv → BN → ReLU     │
└────────────────┬───────────────────┘
                 │
                 ▼  (Reshape to sequence)
┌────────────────────────────────────┐
│     TRANSFORMER MODULE             │
│                                    │
│  Positional Encoding               │
│  Multi-Head Self-Attention (×N)    │
│  Feed-Forward Network              │
│  (Captures long-range dependencies)│
└────────────────┬───────────────────┘
                 │
                 ▼  (Reshape back to 3D)
┌────────────────────────────────────┐
│     3D CNN DECODER                 │
│                                    │
│  Upsample → Concat(Skip) → Conv   │
│  (×3 levels)                       │
│        │                           │
│  1×1×1 Conv → Softmax             │
└────────────────────────────────────┘
        │
        ▼
   Multi-class Segmentation
```

### Key Innovations
- **Hybrid CNN-Transformer**: Combines CNN's strength in local feature extraction with Transformer's ability to model global dependencies
- **3D volumetric processing**: Operates on 3D MRI volumes natively
- **Positional encoding**: Preserves spatial information when converting 3D features to a sequence for the Transformer
- **Efficient design**: Only applies Transformer at the bottleneck (lowest resolution), keeping computational cost manageable

### Performance
- **BraTS 2019**: Dice scores of 88.4% (Whole Tumor), 83.6% (Tumor Core), 78.9% (Enhancing Tumor)
- Demonstrated that global context from Transformers improves segmentation of tumor sub-regions

### Reference
- Wang, W. et al. "TransBTS: Multimodal Brain Tumor Segmentation Using Transformer." *MICCAI 2021*. [DOI:10.1007/978-3-030-87193-2_11](https://doi.org/10.1007/978-3-030-87193-2_11)
- GitHub: [https://github.com/Wenxuan-1119/TransBTS](https://github.com/Wenxuan-1119/TransBTS)

## <span style="color:#1a5276;font-size:22px">4.5 Attention U-Net</span> <a id="sec4_5"></a>

### Overview
**Attention U-Net** enhances the standard U-Net by adding **Attention Gates (AGs)** at the skip connections. Instead of blindly concatenating encoder features, the attention gates learn to suppress irrelevant regions and highlight salient features, improving segmentation accuracy especially for small structures.

### Architecture Flow

```
   Input Image
        │
        ▼
┌────────────────────────────────────┐
│     ENCODER (same as U-Net)        │
│  Conv → Conv → Pool  (×4 levels)  │
│  Each level produces features: Fᵢ  │
└────────────────┬───────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│         BOTTLENECK                 │
└────────────────┬───────────────────┘
                 │
                 ▼
┌────────────────────────────────────────────────────┐
│     DECODER with ATTENTION GATES                   │
│                                                    │
│  For each level i:                                 │
│                                                    │
│    Decoder features (gating signal): gᵢ            │
│    Encoder features (skip):          Fᵢ            │
│              │                        │            │
│              ▼                        ▼            │
│    ┌──────────────────────────────────────┐        │
│    │       ATTENTION GATE                 │        │
│    │                                      │        │
│    │  Wg·gᵢ + Wx·Fᵢ → ReLU → ψ → σ     │        │
│    │          │                            │        │
│    │          ▼                            │        │
│    │   Attention map: αᵢ ∈ [0,1]          │        │
│    │          │                            │        │
│    │   Output: αᵢ × Fᵢ  (reweighted)     │        │
│    └──────────────────────────────────────┘        │
│              │                                     │
│    Concat(αᵢ×Fᵢ, Upsampled gᵢ) → Conv → Conv     │
│                                                    │
└────────────────────────────────────────────────────┘
        │
        ▼
   Segmentation Mask
```

### Key Innovations
- **Soft attention on skip connections**: Learns to weight which spatial regions and features from the encoder are relevant for the current decoding step
- **No additional supervision needed**: Attention is learned implicitly from the segmentation loss
- **Improved small-object segmentation**: Particularly effective for small or variable-size structures (e.g., pancreas, small tumors)
- **Lightweight**: Adds minimal parameters and computation compared to standard U-Net

### Performance
- Significantly improves segmentation of small structures compared to vanilla U-Net
- Widely adopted in medical imaging pipelines as a drop-in improvement

### Reference
- Oktay, O. et al. "Attention U-Net: Learning Where to Look for the Pancreas." *MIDL 2018*. [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)

## <span style="color:#1a5276;font-size:22px">4.6 Model Comparison</span> <a id="sec4_6"></a>

| Model | Year | Venue | Backbone | Key Innovation | BraTS Dice (WT) | Pros | Cons |
|-------|------|-------|----------|----------------|-----------------|------|------|
| **U-Net** | 2015 | MICCAI | CNN | Skip connections, encoder-decoder | ~87% | Simple, proven, easy to train | Limited receptive field, no global context |
| **nnU-Net** | 2021 | Nat. Methods | Auto-configured CNN | Self-configuring framework | ~91% | No manual tuning, robust across tasks | Computationally expensive, long training |
| **Swin UNETR** | 2021 | BrainLes/CVPR | Swin Transformer + CNN | Shifted-window attention, self-supervised pre-training | ~92% | Global context, SOTA on BraTS 2021 | Requires large GPU memory, complex |
| **MedSAM** | 2024 | Nat. Commun. | ViT (SAM) | Foundation model, promptable | ~89%* | Versatile, interactive, multi-modality | Prompt-dependent, 2D only, variable quality |
| **TransBTS** | 2021 | MICCAI | CNN + Transformer | Hybrid CNN-Transformer at bottleneck | ~88% | Global context with manageable cost | Transformer only at bottleneck |
| **Attention U-Net** | 2018 | MIDL | CNN | Attention gates on skip connections | ~88% | Simple improvement over U-Net | Still limited to local features |

*\*MedSAM Dice varies significantly based on prompt quality and target structure.*

### When to Use Which Model

| Scenario | Recommended Model |
|----------|-------------------|
| Quick baseline / limited compute | **U-Net** or **Attention U-Net** |
| Best out-of-the-box performance | **nnU-Net** |
| State-of-the-art with large GPU budget | **Swin UNETR** |
| Interactive / clinical workflow | **MedSAM** |
| Research with 3D volumetric data | **Swin UNETR** or **TransBTS** |
| New dataset with unknown optimal config | **nnU-Net** |

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 5 — Hands-On Implementation</p> <a id="part5"></a>

Now let's build a complete brain tumor segmentation pipeline using U-Net in TensorFlow/Keras.

**Dataset**: LGG MRI Segmentation — 3,929 brain MRI images from 110 patients with lower-grade gliomas, each paired with a manually created FLAIR abnormality segmentation mask.

**Goal**: Train a U-Net to predict segmentation masks from brain MRI images, achieving a **Dice coefficient > 85%**.

## <span style="color:#1a5276;font-size:22px">5.1 Setup & Imports</span> <a id="sec5_1"></a>

We begin by importing the required libraries. This pipeline uses:
- **TensorFlow/Keras** for model building and training
- **scikit-learn** for train/test splitting
- **OpenCV & scikit-image** for image processing
- **matplotlib & seaborn** for visualization

In [ ]:
# import system libs
import os
import time
import random
import pathlib
import itertools
from glob import glob
from tqdm import tqdm_notebook, tnrange

# import data handling tools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
%matplotlib inline
from skimage.color import rgb2gray
from skimage.morphology import label
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow, concatenate_images

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model, load_model, save_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Input, Activation, BatchNormalization, Dropout, Lambda, Conv2D, Conv2DTranspose, MaxPooling2D, concatenate

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print('modules loaded')

## <span style="color:#1a5276;font-size:22px">5.2 Data Loading & Exploration</span> <a id="sec5_2"></a>

### Creating a DataFrame from the Dataset

The dataset is organized as patient directories, each containing MRI images and their corresponding masks. Mask files contain `_mask` in their filename. We create a DataFrame that pairs each image with its mask.

> **Note**: The dataset is hosted on Kaggle at `/kaggle/input/lgg-mri-segmentation/kaggle_3m`.

In [ ]:
# Function to create a DataFrame mapping images to their masks
def create_df(data_dir):
    images_paths = []
    masks_paths = glob(f'{data_dir}/*/*_mask*')

    for i in masks_paths:
        images_paths.append(i.replace('_mask', ''))

    df = pd.DataFrame(data= {'images_paths': images_paths, 'masks_paths': masks_paths})

    return df

# Function to split DataFrame into train (80%), valid (10%), test (10%)
def split_df(df):
    # create train_df
    train_df, dummy_df = train_test_split(df, train_size= 0.8)

    # create valid_df and test_df
    valid_df, test_df = train_test_split(dummy_df, train_size= 0.5)

    return train_df, valid_df, test_df

### Understanding the Data Split

| Split | Proportion | Purpose |
|-------|-----------|---------|
| **Training** | 80% | Model learns from these examples |
| **Validation** | 10% | Monitor performance during training, tune hyperparameters |
| **Test** | 10% | Final unbiased evaluation of model performance |

A common alternative split is 70/15/15. The key principle: the test set must **never** influence model training or hyperparameter selection.

## <span style="color:#1a5276;font-size:22px">5.3 Data Preprocessing & Augmentation</span> <a id="sec5_3"></a>

### Image Generators

We use Keras `ImageDataGenerator` to:
1. **Load images on-the-fly** from disk (memory-efficient)
2. **Apply data augmentation** to training images (not validation/test)
3. **Normalize** pixel values to [0, 1] range
4. **Binarize** masks (threshold at 0.5)

### Why Data Augmentation?

Medical imaging datasets are typically small. Augmentation artificially increases training diversity:

| Augmentation | Effect | Why It Helps |
|-------------|--------|--------------|
| **Rotation** | Slight rotation (±0.2 rad) | Tumors can appear at any angle |
| **Width/Height Shift** | Small translations | Tumor may not be centered |
| **Shear** | Parallelogram distortion | Simulates viewing angle variations |
| **Zoom** | Scale in/out | Tumors vary in size |
| **Horizontal Flip** | Mirror image | Brain has bilateral symmetry |

> **Important**: The **same** augmentation must be applied to both the image and its mask (same random seed ensures this).

In [ ]:
def create_gens(df, aug_dict):
    img_size = (256, 256)
    batch_size = 16  # Reduced for Keras 3 + XLA memory usage on T4 GPU. You can also try 20 or 24.


    img_gen = ImageDataGenerator(**aug_dict)
    msk_gen = ImageDataGenerator(**aug_dict)

    # Create general generator
    image_gen = img_gen.flow_from_dataframe(df, x_col='images_paths', class_mode=None, color_mode='rgb', target_size=img_size,
                                            batch_size=batch_size, save_to_dir=None, save_prefix='image', seed=1)

    mask_gen = msk_gen.flow_from_dataframe(df, x_col='masks_paths', class_mode=None, color_mode='grayscale', target_size=img_size,
                                            batch_size=batch_size, save_to_dir=None, save_prefix= 'mask', seed=1)

    gen = zip(image_gen, mask_gen)

    for (img, msk) in gen:
        img = img / 255
        msk = msk / 255
        msk[msk > 0.5] = 1
        msk[msk <= 0.5] = 0

        yield (img, msk)

### Visualizing Sample Images

Let's visualize some training images with their tumor masks overlaid.

In [ ]:
def show_images(images, masks):
    plt.figure(figsize=(12, 12))
    for i in range(25):
        plt.subplot(5, 5, i+1)
        img_path = images[i]
        mask_path = masks[i]
        # read image and convert it to RGB scale
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # read mask
        mask = cv2.imread(mask_path)
        # show image and mask
        plt.imshow(image)
        plt.imshow(mask, alpha=0.4)

        plt.axis('off')

    plt.tight_layout()
    plt.show()

### Load Data and Create Generators

In [ ]:
# Auto-detect dataset path (varies depending on how the dataset was added on Kaggle)
import os
candidate_paths = [
    '/kaggle/input/lgg-mri-segmentation/kaggle_3m',
    '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m',
    '/kaggle/input/lgg-mri-segmentation',
    '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation',
]
data_dir = None
for p in candidate_paths:
    if os.path.isdir(p):
        data_dir = p
        break
if data_dir is None:
    # Fallback: search for any directory containing '_mask' files
    for root, dirs, files in os.walk('/kaggle/input'):
        if any('_mask' in f for f in files):
            data_dir = os.path.dirname(root)
            break
assert data_dir is not None, "Dataset not found! Please check the dataset path."
print(f"Using dataset path: {data_dir}")

df = create_df(data_dir)
train_df, valid_df, test_df = split_df(df)

print(f"Total images: {len(df)}")
print(f"Training:     {len(train_df)}")
print(f"Validation:   {len(valid_df)}")
print(f"Test:         {len(test_df)}")

tr_aug_dict = dict(rotation_range=0.2,
                            width_shift_range=0.05,
                            height_shift_range=0.05,
                            shear_range=0.05,
                            zoom_range=0.05,
                            horizontal_flip=True,
                            fill_mode='nearest')


train_gen = create_gens(train_df, aug_dict=tr_aug_dict)
valid_gen = create_gens(valid_df, aug_dict={})
test_gen = create_gens(test_df, aug_dict={})

show_images(list(train_df['images_paths']), list(train_df['masks_paths']))

## <span style="color:#1a5276;font-size:22px">5.4 Build U-Net Model</span> <a id="sec5_4"></a>

Now we implement the U-Net architecture in TensorFlow/Keras. Our implementation follows the structure described in Part 3:

- **Input**: 256×256×3 (RGB MRI images)
- **Encoder**: 4 levels with [64, 128, 256, 512] filters
- **Bottleneck**: 1024 filters
- **Decoder**: 4 levels mirroring the encoder
- **Output**: 256×256×1 (binary mask, sigmoid activation)

Each encoder/decoder block uses:
- Two 3×3 convolutions with ReLU activation
- Batch normalization after the second convolution
- Max pooling (encoder) / Transposed convolution (decoder)

In [ ]:
def unet(input_size=(256, 256, 3)):
    inputs = Input(input_size)

    # First DownConvolution / Encoder Leg will begin, so start with Conv2D
    conv1 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(inputs)
    bn1 = Activation("relu")(conv1)
    conv1 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(bn1)
    bn1 = BatchNormalization(axis=3)(conv1)
    bn1 = Activation("relu")(bn1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(bn1)

    conv2 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(pool1)
    bn2 = Activation("relu")(conv2)
    conv2 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(bn2)
    bn2 = BatchNormalization(axis=3)(conv2)
    bn2 = Activation("relu")(bn2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(bn2)

    conv3 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(pool2)
    bn3 = Activation("relu")(conv3)
    conv3 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(bn3)
    bn3 = BatchNormalization(axis=3)(conv3)
    bn3 = Activation("relu")(bn3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(bn3)

    conv4 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(pool3)
    bn4 = Activation("relu")(conv4)
    conv4 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(bn4)
    bn4 = BatchNormalization(axis=3)(conv4)
    bn4 = Activation("relu")(bn4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(bn4)

    conv5 = Conv2D(filters=1024, kernel_size=(3, 3), padding="same")(pool4)
    bn5 = Activation("relu")(conv5)
    conv5 = Conv2D(filters=1024, kernel_size=(3, 3), padding="same")(bn5)
    bn5 = BatchNormalization(axis=3)(conv5)
    bn5 = Activation("relu")(bn5)

    # Now UpConvolution / Decoder Leg will begin, so start with Conv2DTranspose
    # The gray arrows (in the above image) indicate the skip connections that concatenate
    # the encoder feature map with the decoder, which helps the backward flow of gradients
    # for improved training.
    # After every concatenation we again apply two consecutive regular convolutions so that
    # the model can learn to assemble a more precise output.

    up6 = concatenate([Conv2DTranspose(512, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn5), conv4], axis=3)
    conv6 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(up6)
    bn6 = Activation("relu")(conv6)
    conv6 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(bn6)
    bn6 = BatchNormalization(axis=3)(conv6)
    bn6 = Activation("relu")(bn6)

    up7 = concatenate([Conv2DTranspose(256, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn6), conv3], axis=3)
    conv7 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(up7)
    bn7 = Activation("relu")(conv7)
    conv7 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(bn7)
    bn7 = BatchNormalization(axis=3)(conv7)
    bn7 = Activation("relu")(bn7)

    up8 = concatenate([Conv2DTranspose(128, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn7), conv2], axis=3)
    conv8 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(up8)
    bn8 = Activation("relu")(conv8)
    conv8 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(bn8)
    bn8 = BatchNormalization(axis=3)(conv8)
    bn8 = Activation("relu")(bn8)

    up9 = concatenate([Conv2DTranspose(64, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn8), conv1], axis=3)
    conv9 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(up9)
    bn9 = Activation("relu")(conv9)
    conv9 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(bn9)
    bn9 = BatchNormalization(axis=3)(conv9)
    bn9 = Activation("relu")(bn9)

    conv10 = Conv2D(filters=1, kernel_size=(1, 1), activation="sigmoid")(bn9)

    return Model(inputs=[inputs], outputs=[conv10])

### Understanding Each Layer

Let's trace what happens to a 256×256×3 input:

| Layer | Output Shape | # Parameters | Notes |
|-------|-------------|-------------|-------|
| Input | 256×256×3 | 0 | RGB MRI image |
| Encoder L1 (2×Conv64) | 256×256×64 | ~37K | First features: edges, textures |
| MaxPool | 128×128×64 | 0 | Spatial reduction by 2× |
| Encoder L2 (2×Conv128) | 128×128×128 | ~222K | Shape patterns |
| MaxPool | 64×64×128 | 0 | |
| Encoder L3 (2×Conv256) | 64×64×256 | ~886K | Complex features |
| MaxPool | 32×32×256 | 0 | |
| Encoder L4 (2×Conv512) | 32×32×512 | ~3.5M | High-level semantics |
| MaxPool | 16×16×512 | 0 | |
| **Bottleneck** (2×Conv1024) | **16×16×1024** | **~14M** | **Deepest representation** |
| UpConv + Skip + 2×Conv512 | 32×32×512 | ~11M | Recover spatial detail |
| UpConv + Skip + 2×Conv256 | 64×64×256 | ~2.8M | |
| UpConv + Skip + 2×Conv128 | 128×128×128 | ~700K | |
| UpConv + Skip + 2×Conv64 | 256×256×64 | ~175K | |
| **Output** (Conv 1×1) | **256×256×1** | **65** | **Sigmoid → binary mask** |
| **Total** | | **~34.5M** | |

## <span style="color:#1a5276;font-size:22px">5.5 Loss Functions & Metrics</span> <a id="sec5_5"></a>

We define three key functions for training and evaluation:

1. **Dice Coefficient** — Primary metric for overlap quality
2. **Dice Loss** — Negative Dice coefficient (we minimize this)
3. **IoU Coefficient** — Secondary metric for evaluation

The `smooth` parameter (default=100) prevents division by zero and stabilizes training when both prediction and ground truth are empty (no tumor present).

In [ ]:
# Dice coefficient: measures overlap between prediction and ground truth
def dice_coef(y_true, y_pred, smooth=100):
    y_true_flatten = K.flatten(y_true)
    y_pred_flatten = K.flatten(y_pred)

    intersection = K.sum(y_true_flatten * y_pred_flatten)
    union = K.sum(y_true_flatten) + K.sum(y_pred_flatten)
    return (2 * intersection + smooth) / (union + smooth)

# Dice loss: negative Dice coefficient (to minimize)
def dice_loss(y_true, y_pred, smooth=100):
    return -dice_coef(y_true, y_pred, smooth)

# IoU coefficient: Intersection over Union
def iou_coef(y_true, y_pred, smooth=100):
    intersection = K.sum(y_true * y_pred)
    sum = K.sum(y_true + y_pred)
    iou = (intersection + smooth) / (sum - intersection + smooth)
    return iou

### Visualizing How Dice and IoU Relate

For a given prediction:
- **Perfect overlap**: Dice = 1.0, IoU = 1.0
- **50% overlap**: Dice ≈ 0.67, IoU = 0.50
- **No overlap**: Dice = 0.0, IoU = 0.0

The Dice coefficient is always ≥ IoU for the same prediction. Both are valid metrics, but Dice is the standard for the BraTS challenge and most medical segmentation benchmarks.

## <span style="color:#1a5276;font-size:22px">5.6 Compile & Train</span> <a id="sec5_6"></a>

### Training Configuration

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| **Optimizer** | Adamax (lr=0.001) | Variant of Adam that is more robust to large gradients |
| **Loss** | Dice Loss | Directly optimizes the metric we care about |
| **Metrics** | Accuracy, IoU, Dice | Track multiple perspectives on performance |
| **Epochs** | 120 | Sufficient for convergence on this dataset |
| **Batch Size** | 40 | Balances memory usage and gradient stability |
| **Callbacks** | ModelCheckpoint | Saves the best model (lowest validation loss) |

### Why Adamax?

Adamax is a variant of Adam based on the infinity norm. It is sometimes more stable than Adam for models with embeddings or when gradients have high variance. For this segmentation task, it provides smooth convergence.

In [ ]:
model = unet()
model.compile(Adamax(learning_rate= 0.001), loss= dice_loss, metrics= ['accuracy', iou_coef, dice_coef])

model.summary()

In [ ]:
epochs = 120
batch_size = 16  # Reduced for Keras 3 + XLA memory usage on T4 GPU. You can also try 20 or 24.
callbacks = [ModelCheckpoint('unet.keras', verbose=0, save_best_only=True)]

history = model.fit(train_gen,
                    steps_per_epoch=len(train_df) // batch_size,
                    epochs=epochs,
                    verbose=1,
                    callbacks=callbacks,
                    validation_data = valid_gen,
                    validation_steps=len(valid_df) // batch_size)

## <span style="color:#1a5276;font-size:22px">5.7 Training History Visualization</span> <a id="sec5_7"></a>

Visualizing training curves is essential for diagnosing:
- **Overfitting**: Training metrics improve but validation metrics plateau or degrade
- **Underfitting**: Both training and validation metrics are poor
- **Convergence**: Both curves plateau at similar values

In [ ]:
def plot_training(hist):
    '''
    This function take training model and plot history of accuracy and losses with the best epoch in both of them.
    '''

    # Define needed variables
    tr_acc = hist.history['accuracy']
    tr_iou = hist.history['iou_coef']
    tr_dice = hist.history['dice_coef']
    tr_loss = hist.history['loss']

    val_acc = hist.history['val_accuracy']
    val_iou = hist.history['val_iou_coef']
    val_dice = hist.history['val_dice_coef']
    val_loss = hist.history['val_loss']

    index_acc = np.argmax(val_acc)
    acc_highest = val_acc[index_acc]
    index_iou = np.argmax(val_iou)
    iou_highest = val_iou[index_iou]
    index_dice = np.argmax(val_dice)
    dice_highest = val_dice[index_dice]
    index_loss = np.argmin(val_loss)
    val_lowest = val_loss[index_loss]

    Epochs = [i+1 for i in range(len(tr_acc))]

    acc_label = f'best epoch= {str(index_acc + 1)}'
    iou_label = f'best epoch= {str(index_iou + 1)}'
    dice_label = f'best epoch= {str(index_dice + 1)}'
    loss_label = f'best epoch= {str(index_loss + 1)}'

    # Plot training history
    plt.figure(figsize= (20, 20))
    plt.style.use('fivethirtyeight')

    # Training Accuracy
    plt.subplot(2, 2, 1)
    plt.plot(Epochs, tr_acc, 'r', label= 'Training Accuracy')
    plt.plot(Epochs, val_acc, 'g', label= 'Validation Accuracy')
    plt.scatter(index_acc + 1 , acc_highest, s= 150, c= 'blue', label= acc_label)
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Training IoU
    plt.subplot(2, 2, 2)
    plt.plot(Epochs, tr_iou, 'r', label= 'Training IoU')
    plt.plot(Epochs, val_iou, 'g', label= 'Validation IoU')
    plt.scatter(index_iou + 1 , iou_highest, s= 150, c= 'blue', label= iou_label)
    plt.title('Training and Validation IoU Coefficient')
    plt.xlabel('Epochs')
    plt.ylabel('IoU')
    plt.legend()

    # Training Dice
    plt.subplot(2, 2, 3)
    plt.plot(Epochs, tr_dice, 'r', label= 'Training Dice')
    plt.plot(Epochs, val_dice, 'g', label= 'Validation Dice')
    plt.scatter(index_dice + 1 , dice_highest, s= 150, c= 'blue', label= dice_label)
    plt.title('Training and Validation Dice Coefficient')
    plt.xlabel('Epochs')
    plt.ylabel('Dice')
    plt.legend()

    # Training Loss
    plt.subplot(2, 2, 4)
    plt.plot(Epochs, tr_loss, 'r', label= 'Training loss')
    plt.plot(Epochs, val_loss, 'g', label= 'Validation loss')
    plt.scatter(index_loss + 1, val_lowest, s= 150, c= 'blue', label= loss_label)
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout
    plt.show()

In [ ]:
plot_training(history)

## <span style="color:#1a5276;font-size:22px">5.8 Model Evaluation</span> <a id="sec5_8"></a>

We evaluate the model on all three splits to check for overfitting:

| Expected Pattern | Interpretation |
|-----------------|----------------|
| Train ≈ Valid ≈ Test | Good generalization |
| Train >> Valid ≈ Test | Overfitting — model memorizes training data |
| Train ≈ Valid >> Test | Data leakage or distribution shift |

In [ ]:
ts_length = len(test_df)
test_batch_size = max(sorted([ts_length // n for n in range(1, ts_length + 1) if ts_length%n == 0 and ts_length/n <= 80]))
test_steps = ts_length // test_batch_size

train_score = model.evaluate(train_gen, steps= test_steps, verbose= 1)
valid_score = model.evaluate(valid_gen, steps= test_steps, verbose= 1)
test_score = model.evaluate(test_gen, steps= test_steps, verbose= 1)


print("Train Loss: ", train_score[0])
print("Train Accuracy: ", train_score[1])
print("Train IoU: ", train_score[2])
print("Train Dice: ", train_score[3])
print('-' * 20)

print("Valid Loss: ", valid_score[0])
print("Valid Accuracy: ", valid_score[1])
print("Valid IoU: ", valid_score[2])
print("Valid Dice: ", valid_score[3])
print('-' * 20)

print("Test Loss: ", test_score[0])
print("Test Accuracy: ", test_score[1])
print("Test IoU: ", test_score[2])
print("Test Dice: ", test_score[3])

### Interpreting the Results

**Expected results** (approximate, may vary due to random splits):
- **Dice Coefficient**: ~87-90% on test set
- **IoU**: ~77-83% on test set
- **Pixel Accuracy**: ~99.7% (high due to class imbalance — most pixels are background)

The gap between Dice (87-90%) and Accuracy (99.7%) illustrates why Dice/IoU are better metrics for segmentation tasks with class imbalance.

## <span style="color:#1a5276;font-size:22px">5.9 Prediction Visualization</span> <a id="sec5_9"></a>

Let's visualize predictions on test images to qualitatively assess the model. For each sample, we show:
1. **Original MRI image** — the input
2. **Ground truth mask** — the manually annotated tumor region
3. **Predicted mask** — our model's prediction (thresholded at 0.5)

In [ ]:
for _ in range(20):
    index = np.random.randint(1, len(test_df.index))
    img = cv2.imread(test_df['images_paths'].iloc[index])
    img = cv2.resize(img, (256, 256))
    img = img/255
    img = img[np.newaxis, :, :, : ]

    predicted_img = model.predict(img)

    plt.figure(figsize=(12, 12))

    plt.subplot(1, 3, 1)
    plt.imshow(np.squeeze(img))
    plt.axis('off')
    plt.title('Original Image')

    plt.subplot(1, 3, 2)
    plt.imshow(np.squeeze(cv2.imread(test_df['masks_paths'].iloc[index])))
    plt.axis('off')
    plt.title('Original Mask')

    plt.subplot(1, 3, 3)
    plt.imshow(np.squeeze(predicted_img) > 0.5 )
    plt.title('Prediction')
    plt.axis('off')

    plt.show()

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 6 — Summary & Next Steps</p> <a id="part6"></a>

### What We Covered

| Topic | Key Takeaway |
|-------|-------------|
| **Brain tumors** | Gliomas are the most common primary brain tumors; automated segmentation aids diagnosis and treatment |
| **Image segmentation** | Pixel-level classification; semantic segmentation assigns a class to every pixel |
| **Metrics** | Dice and IoU are preferred over pixel accuracy for imbalanced segmentation tasks |
| **U-Net** | Encoder-decoder with skip connections; designed for biomedical segmentation with limited data |
| **SOTA models** | nnU-Net (self-configuring), Swin UNETR (transformer-based), MedSAM (foundation model), TransBTS (hybrid), Attention U-Net (attention gates) |
| **Implementation** | Built a U-Net in TF/Keras achieving ~89% Dice on the LGG MRI dataset |

### Next Steps

1. **Try 3D segmentation**: Extend to volumetric models using the BraTS dataset with multi-modal MRI (T1, T1ce, T2, FLAIR)
2. **Experiment with architectures**: Implement Attention U-Net or use `segmentation_models` library for pre-trained encoders
3. **Try nnU-Net**: Install and run the nnU-Net framework on your data for automated configuration
4. **Explore transfer learning**: Use pre-trained encoders (ResNet, EfficientNet) as the U-Net backbone
5. **Post-processing**: Apply connected component analysis and morphological operations to clean up predictions
6. **Multi-class segmentation**: Segment tumor sub-regions (enhancing tumor, tumor core, whole tumor)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">References</p> <a id="refs"></a>

1. **U-Net**: Ronneberger, O., Fischer, P., & Brox, T. (2015). "U-Net: Convolutional Networks for Biomedical Image Segmentation." *MICCAI 2015*. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)

2. **nnU-Net**: Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. (2021). "nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation." *Nature Methods*, 18, 203–211. [DOI:10.1038/s41592-020-01008-z](https://doi.org/10.1038/s41592-020-01008-z) | [GitHub](https://github.com/MIC-DKFZ/nnUNet)

3. **Swin UNETR**: Hatamizadeh, A. et al. (2022). "Swin UNETR: Swin Transformers for Semantic Segmentation of Brain Tumours in MRI Images." *BrainLes Workshop, MICCAI 2021*. [arXiv:2201.01266](https://arxiv.org/abs/2201.01266) | [GitHub](https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/swin_unetr_brats21_segmentation_3d.ipynb)

4. **Swin UNETR Pre-training**: Tang, Y. et al. (2022). "Self-Supervised Pre-Training of Swin Transformers for 3D Medical Image Analysis." *CVPR 2022*. [arXiv:2111.14791](https://arxiv.org/abs/2111.14791)

5. **MedSAM**: Ma, J. et al. (2024). "Segment Anything in Medical Images." *Nature Communications*, 15, 654. [DOI:10.1038/s41467-024-44824-z](https://doi.org/10.1038/s41467-024-44824-z) | [GitHub](https://github.com/bowang-lab/MedSAM) | [Brain-specific](https://github.com/vpulab/med-sam-brain)

6. **TransBTS**: Wang, W. et al. (2021). "TransBTS: Multimodal Brain Tumor Segmentation Using Transformer." *MICCAI 2021*. [DOI:10.1007/978-3-030-87193-2_11](https://doi.org/10.1007/978-3-030-87193-2_11) | [GitHub](https://github.com/Wenxuan-1119/TransBTS)

7. **Attention U-Net**: Oktay, O. et al. (2018). "Attention U-Net: Learning Where to Look for the Pancreas." *MIDL 2018*. [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)

8. **Regional-Aware U-Net**: Abbas et al. [GitHub](https://github.com/abbas695/Regional_aware_U-NET)

9. **LGG MRI Dataset**: Buda, M., Saha, A., & Mazurowski, M.A. (2019). "Association of genomic subtypes of lower-grade gliomas with shape features automatically extracted by a deep learning algorithm." *Computers in Biology and Medicine*, 109, 218–225. | [Kaggle](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)

10. **BraTS Challenge**: Menze, B.H. et al. (2015). "The Multimodal Brain Tumor Image Segmentation Benchmark (BRATS)." *IEEE TMI*, 34(10), 1993–2024. [DOI:10.1109/TMI.2014.2377694](https://doi.org/10.1109/TMI.2014.2377694)